# Sélection et vérification 

In [ ]:
def compute_score(item):
    authors = item.get('authors', [])
    references = item.get('references', [])
    citations = item.get('n_citation', 0)

    n_authors = len(authors)
    n_refs = len(references)

    collaboration_bonus = 1 if n_authors >= 2 else 0

    return (
        0.4 * citations +
        0.3 * n_refs +
        0.2 * n_authors +
        0.1 * collaboration_bonus
    )

In [ ]:
# quotas (≈ 1M total)
year_quota = {
    2015: 95000,
    2016: 95000,
    2017: 100000,
    2018: 100000,
    2019: 100000,
    2020: 100000,
    2021: 100000,
    2022: 100000,
    2023: 90000,
    2024: 90000,
    2025: 30000
}

selected_papers = set()

print("📊 Sélection des meilleurs papers...")

for year, papers in year_papers.items():
    papers_sorted = sorted(papers, key=lambda x: x[1], reverse=True)

    quota = year_quota.get(year, 0)

    for p_id, _ in papers_sorted[:quota]:
        selected_papers.add(p_id)

print(f"✅ Papers sélectionnés : {len(selected_papers):,}")
import json
from tqdm import tqdm

OUTPUT_PATH = "/kaggle/working/selected_papers_full.jsonl"
INPUT_FILE = '/kaggle/input/datasets/ayamhiri/dblp-citation-network-v18/DBLP-Citation-network-V18.jsonl'

print("💾 Sauvegarde des papers complets...")

with open(INPUT_FILE, "r", encoding="utf-8") as f_in, \
     open(OUTPUT_PATH, "w", encoding="utf-8") as f_out:

    for line in tqdm(f_in):
        try:
            item = json.loads(line)
        except:
            continue

        if item.get("id") in selected_papers:
            f_out.write(json.dumps(item) + "\n")

print(f"✅ Sauvegardé : {len(selected_papers):,} papers")


In [ ]:
count = 0

with open("/kaggle/working/selected_papers_full.jsonl", "r", encoding="utf-8") as f:
    for _ in f:
        count += 1

print(f"📦 Nombre de papers dans le fichier : {count:,}")

In [ ]:
import json

with open("/kaggle/working/selected_papers_full.jsonl", "r", encoding="utf-8") as f:
    line = f.readline()
    item = json.loads(line)

print("🔎 Exemple de paper :\n")
print(json.dumps(item, indent=2, ensure_ascii=False))

In [ ]:
import json
from tqdm import tqdm
from itertools import combinations  # ✅ AJOUT ICI
author_map, paper_map = {}, {}
p_count, a_count = 0, 0
OUTPUT_DIR ='/kaggle/working'
INPUT_FILE='/kaggle/working/selected_papers_full.jsonl'
with open(INPUT_FILE, 'r', encoding='utf-8') as f, \
     open(f'{OUTPUT_DIR}/nodes_papers.jsonl', 'w') as f_p, \
     open(f'{OUTPUT_DIR}/nodes_authors.jsonl', 'w') as f_a, \
     open(f'{OUTPUT_DIR}/edges_author_paper.jsonl', 'w') as f_ap, \
     open(f'{OUTPUT_DIR}/edges_author_author.jsonl', 'w') as f_aa, \
     open(f'{OUTPUT_DIR}/edges_paper_paper.jsonl', 'w') as f_pp:

    for line in tqdm(f, desc="Construction graphe"):
        try:
            item = json.loads(line)
        except:
            continue

        p_orig_id = item.get('id')
        year = item.get('year')

        if p_orig_id not in selected_papers:
            continue

        try:
            year = int(year)
        except:
            continue

        # --- PAPER ---
        if p_orig_id not in paper_map:
            paper_map[p_orig_id] = p_count
            f_p.write(json.dumps({
                'p_idx': p_count,
                'year': year,
                'title': (item.get('title') or '').strip(),
                'abstract': (item.get('abstract') or '').strip(),
                'keywords': item.get('keywords') or [],
                'venue': (item.get('venue') or '').strip(),
            }) + '\n')
            p_count += 1

        curr_p_idx = paper_map[p_orig_id]
        curr_auth_indices = []

        # --- AUTHORS ---
        for auth in item.get('authors', []):
            a_id = (auth.get('id') or '').strip() or (auth.get('name') or '').strip()
            if not a_id:
                continue

            if a_id not in author_map:
                author_map[a_id] = a_count
                f_a.write(json.dumps({
                    'a_idx': a_count,
                    'name': auth.get('name'),
                    'org': auth.get('org'),
                }) + '\n')
                a_count += 1

            curr_a_idx = author_map[a_id]
            curr_auth_indices.append(curr_a_idx)

            # A-P
            f_ap.write(json.dumps({
                'src': curr_a_idx,
                'dst': curr_p_idx,
                'year': year
            }) + '\n')

        # --- A-A ---
        if len(curr_auth_indices) > 1:
            for a1, a2 in combinations(sorted(curr_auth_indices), 2):
                f_aa.write(json.dumps({
                    'a1': a1,
                    'a2': a2,
                    'year': year
                }) + '\n')

        # --- P-P ---
        for ref_id in item.get('references', []):
            if ref_id in selected_papers:
                f_pp.write(json.dumps({
                    'src': curr_p_idx,
                    'dst': paper_map.get(ref_id, -1),
                    'year': year
                }) + '\n')

print(f"✅ {p_count:,} papers | {a_count:,} authors")

In [ ]:
import os
import json

BASE_DIR = "/kaggle/working"

FILES = {
    "papers": f"{BASE_DIR}/selected_papers_full.jsonl",
    "authors": f"{BASE_DIR}/nodes_authors.jsonl",
    "ap_edges": f"{BASE_DIR}/edges_author_paper.jsonl",
    "aa_edges": f"{BASE_DIR}/edges_author_author.jsonl",
    "pp_edges": f"{BASE_DIR}/edges_paper_paper.jsonl",
}

# =========================
# 1. CHECK EXISTENCE
# =========================
print("🔍 Vérification des fichiers :\n")

for name, path in FILES.items():
    exists = os.path.exists(path)
    print(f"{name:<10} : {'✅' if exists else '❌'} {path}")

# =========================
# 2. COUNT LINES
# =========================
print("\n📊 Nombre de lignes :\n")

def count_lines(path):
    if not os.path.exists(path):
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

for name, path in FILES.items():
    print(f"{name:<10} : {count_lines(path):,}")

# =========================
# 3. SAMPLE PAPER
# =========================
print("\n🔎 Exemple paper :\n")

if os.path.exists(FILES["papers"]):
    with open(FILES["papers"], "r", encoding="utf-8") as f:
        line = f.readline()
        item = json.loads(line)
        print(json.dumps(item, indent=2, ensure_ascii=False))

# =========================
# 4. CHECK FEATURES
# =========================
print("\n📦 Features détectées :\n")

features = set()

if os.path.exists(FILES["papers"]):
    with open(FILES["papers"], "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            features.update(item.keys())
            break

print(features)

# =========================
# 5. CHECK EDGES CONSISTENCY
# =========================
print("\n🔗 Vérification des edges :\n")

def load_edges(path, keys):
    edges = []
    if not os.path.exists(path):
        return edges
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            item = json.loads(line)
            edges.append(tuple(item[k] for k in keys))
    return edges

ap_edges = load_edges(FILES["ap_edges"], ("src", "dst"))
aa_edges = load_edges(FILES["aa_edges"], ("a1", "a2"))
pp_edges = load_edges(FILES["pp_edges"], ("src", "dst"))

print(f"AP edges : {len(ap_edges):,}")
print(f"AA edges : {len(aa_edges):,}")
print(f"PP edges : {len(pp_edges):,}")

# =========================
# 6. SANITY CHECK
# =========================
print("\n⚠️ Sanity checks :\n")

if len(ap_edges) == 0:
    print("❌ Pas d'edges author-paper détectés")

if len(aa_edges) == 0:
    print("❌ Pas d'edges author-author détectés")

if len(pp_edges) == 0:
    print("❌ Pas d'edges paper-paper détectés")

print("\n✅ Vérification terminée")

In [ ]:
import os
import json

BASE_DIR = "/kaggle/working"

PAPERS_FILE = f"{BASE_DIR}/nodes_papers.jsonl"
AUTHORS_FILE = f"{BASE_DIR}/nodes_authors.jsonl"

# =========================
# COMPTER LES LIGNES
# =========================
def count_lines(path):
    if not os.path.exists(path):
        return 0
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f)

print("📊 Nombre de nodes :\n")
print(f"Papers  : {count_lines(PAPERS_FILE):,}")
print(f"Authors : {count_lines(AUTHORS_FILE):,}")

# =========================
# AFFICHER UN EXEMPLE PAPER
# =========================
print("\n🔎 Exemple paper :\n")

if os.path.exists(PAPERS_FILE):
    with open(PAPERS_FILE, "r", encoding="utf-8") as f:
        item = json.loads(f.readline())
        print(json.dumps(item, indent=2, ensure_ascii=False))

# =========================
# AFFICHER UN EXEMPLE AUTHOR
# =========================
print("\n🔎 Exemple author :\n")

if os.path.exists(AUTHORS_FILE):
    with open(AUTHORS_FILE, "r", encoding="utf-8") as f:
        item = json.loads(f.readline())
        print(json.dumps(item, indent=2, ensure_ascii=False))

# =========================
# EXTRA CHECK
# =========================
print("\n⚠️ Vérification rapide :\n")

if count_lines(PAPERS_FILE) == 0:
    print("❌ Aucun paper node détecté")

if count_lines(AUTHORS_FILE) == 0:
    print("❌ Aucun author node détecté")

print("\n✅ Vérification terminée")

## Nettoyage & Décomposition (nodes + edges)
**Relations extraites :**
- `nodes_papers` : id, title, abstract, keywords, venue, year
- `nodes_authors` : id, name, org
- `edges_author_paper` : auteur → paper (writes)
- `edges_author_author` : auteur ↔ auteur (coauthor)
- `edges_paper_paper` : paper → paper (citations)

In [ ]:
import os
import json
from collections import Counter

BASE_DIR = "/kaggle/working"

FILES = {
    "AP": f"{BASE_DIR}/edges_author_paper.jsonl",
    "AA": f"{BASE_DIR}/edges_author_author.jsonl",
    "PP": f"{BASE_DIR}/edges_paper_paper.jsonl",
}

# =========================
# FONCTION DE LECTURE
# =========================
def load_edges(path, keys):
    edges = []
    if not os.path.exists(path):
        return edges
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                item = json.loads(line)
                edges.append(tuple(item[k] for k in keys))
            except:
                continue
    return edges

# =========================
# ANALYSE AP
# =========================
print("🔗 EDGES AUTHOR → PAPER\n")

ap = load_edges(FILES["AP"], ("src", "dst", "year"))

print(f"Nombre d'edges AP : {len(ap):,}")

if len(ap) > 0:
    print("\nExemple :")
    print(ap[0])

    years = [e[2] for e in ap]
    print("\nDistribution des années :")
    print(Counter(years))

# =========================
# ANALYSE AA
# =========================
print("\n🔗 EDGES AUTHOR → AUTHOR\n")

aa = load_edges(FILES["AA"], ("a1", "a2", "year"))

print(f"Nombre d'edges AA : {len(aa):,}")

if len(aa) > 0:
    print("\nExemple :")
    print(aa[0])

    years = [e[2] for e in aa]
    print("\nDistribution des années :")
    print(Counter(years))

# =========================
# ANALYSE PP
# =========================
print("\n🔗 EDGES PAPER → PAPER\n")

pp = load_edges(FILES["PP"], ("src", "dst", "year"))

print(f"Nombre d'edges PP : {len(pp):,}")

if len(pp) > 0:
    print("\nExemple :")
    print(pp[0])

    years = [e[2] for e in pp]
    print("\nDistribution des années :")
    print(Counter(years))

# =========================
# SANITY CHECK
# =========================
print("\n⚠️ CHECKS IMPORTANTS\n")

def check_invalid(edges, name):
    invalid = [e for e in edges if -1 in e or None in e]
    print(f"{name} → edges invalides : {len(invalid)}")

check_invalid(ap, "AP")
check_invalid(aa, "AA")
check_invalid(pp, "PP")

print("\n✅ Vérification terminée")

## Embedding (SentenceTransformer sur title + abstract + keywords + venue)

✅ MAX_LENGTH = 128 

In [ ]:
import os, json, math, shutil
import numpy as np
import torch
from tqdm import tqdm
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION KAGGLE ---
INPUT_FILE = '/kaggle/working/nodes_papers.jsonl'
OUTPUT_DIR = '/kaggle/working/data/processed/embeddings'
OUT_NPY = f'{OUTPUT_DIR}/paper_embeddings.npy'

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── VERIFICATION DES FICHIERS ──────────────────────────────────
# Sur Kaggle, on travaille en local. On ne copie pas depuis Drive ici.
if os.path.exists(OUT_NPY):
    print('✅ Fichier embeddings existant trouvé.')
else:
    print('⚙️ Calcul embeddings (Processus lourd)...')

# --- PARAMÈTRES MODÈLE ---
MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
BATCH_SIZE = 512  # Boosté pour le GPU de Kaggle
MAX_LENGTH = 128
EMB_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Device : {EMB_DEVICE}')

# 1. Charger le modèle
model_emb = SentenceTransformer(MODEL_NAME, device=EMB_DEVICE)
model_emb.max_seq_length = MAX_LENGTH

# 2. Compter le nombre total de papiers
if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f"Le fichier {INPUT_FILE} est introuvable. Vérifie la cellule précédente.")

num_papers = sum(1 for _ in open(INPUT_FILE, 'r', encoding='utf-8'))
print(f'📚 Papers à traiter : {num_papers:,}')

# 3. Générateur de texte (Optimisé pour ne pas saturer la RAM)
def text_generator():
    with open(INPUT_FILE, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                obj = json.loads(line)
                parts = [
                    (obj.get('title') or '').strip(),
                    (obj.get('abstract') or '').strip(),
                    (obj.get('venue') or '').strip(),
                    ' '.join(obj.get('keywords', [])) if isinstance(obj.get('keywords'), list) else str(obj.get('keywords') or '')
                ]
                yield ' '.join([p for p in parts if p])
            except:
                yield "" # Ligne corrompue

# 4. Initialiser ou reprendre embeddings (Mémoire mappée sur disque)
dim = 384

if os.path.exists(OUT_NPY):
    print("♻️ Reprise depuis fichier existant...")
    # On ouvre en mode r+ (lecture/écriture)
    embeddings = np.memmap(OUT_NPY, dtype=np.float32, mode='r+', shape=(num_papers, dim))
    
    # Détection de la progression (on cherche la première ligne vide/nulle)
    # Note: Cette étape peut prendre 1-2 min pour 3.7M de lignes
    norms = np.linalg.norm(embeddings[::100], axis=1) # On check 1 ligne sur 100 pour aller vite
    valid_batches = np.count_nonzero(norms)
    start_idx = valid_batches * 100
    print(f"🔁 Reprise approximative à partir de l'index : {start_idx}")
else:
    print("🆕 Création d'une nouvelle matrice d'embeddings...")
    embeddings = np.memmap(OUT_NPY, dtype=np.float32, mode='w+', shape=(num_papers, dim))
    start_idx = 0

# 5. Boucle d'encodage
gen = text_generator()

# Sauter les éléments déjà traités
print("⏩ Saut des éléments déjà encodés...")
for _ in tqdm(range(start_idx), desc="Skipping"):
    try:
        next(gen)
    except StopIteration:
        break

# Encodage par batches
for i in tqdm(range(start_idx, num_papers, BATCH_SIZE), desc="Encoding"):
    batch_texts = []
    for _ in range(BATCH_SIZE):
        try:
            batch_texts.append(next(gen))
        except StopIteration:
            break

    if not batch_texts:
        break

    # Calcul des vecteurs
    vecs = model_emb.encode(
        batch_texts,
        batch_size=len(batch_texts),
        show_progress_bar=False,
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    # Écriture directe sur le disque (memmap)
    embeddings[i : i + len(vecs)] = vecs.astype(np.float32)
    
    # Flush périodique pour ne pas perdre de données si crash
    if i % (BATCH_SIZE * 10) == 0:
        embeddings.flush()

print(f'✅ Encodage terminé. Fichier sauvegardé dans : {OUT_NPY}')
print(f'📊 Shape finale : {embeddings.shape}')

## Construction des snapshots temporels (HeteroData PyG)

In [ ]:
!pip install torch-geometric torch-scatter torch-sparse torch-cluster torch-spline-conv -q



In [ ]:
import os, json
import numpy as np
import torch
from tqdm import tqdm
from torch_geometric.data import HeteroData

# --- CONFIGURATION ---
YEAR_START   = 2015
YEAR_END     = 2025
AUTHOR_DIM   = 32

BASE_DIR     = '/kaggle/working'
SNAPSHOT_DIR = f'{BASE_DIR}/processed/snapshots'
EMB_PATH     = f'{BASE_DIR}/data/processed/embeddings/paper_embeddings.npy'

AP_PATH      = f'{BASE_DIR}/edges_author_paper.jsonl'
AA_PATH      = f'{BASE_DIR}/edges_author_author.jsonl'
PP_PATH      = f'{BASE_DIR}/edges_paper_paper.jsonl'

os.makedirs(SNAPSHOT_DIR, exist_ok=True)

# =========================
# ✅ LOAD EMBEDDINGS (SAFE)
# =========================
print('⚙️ Chargement embeddings (AUTO DETECT)...')

if not os.path.exists(EMB_PATH):
    raise FileNotFoundError(f"❌ Embeddings introuvables : {EMB_PATH}")

# Taille fichier
file_size = os.path.getsize(EMB_PATH)

# float32 = 4 bytes
dim = 384
num_rows = file_size // (4 * dim)

print(f"📊 Taille fichier : {file_size}")
print(f"📊 Rows détectés : {num_rows}")

paper_embeddings = np.memmap(
    EMB_PATH,
    dtype='float32',
    mode='r',
    shape=(num_rows, dim)
)

N_ROWS, N_COLS = paper_embeddings.shape

print(f'✅ Embeddings chargés : {paper_embeddings.shape}')

# =========================
# UTILS
# =========================
def read_jsonl(path):
    if not os.path.exists(path):
        print(f"⚠️ Manquant : {path}")
        return []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            yield json.loads(line)

# =========================
# BUILD SNAPSHOT
# =========================
def build_snapshot(year):
    ap_edges, aa_edges, pp_edges = [], [], []
    authors_set, papers_set = set(), set()

    # --- AUTHOR-PAPER ---
    for e in read_jsonl(AP_PATH):
        if str(e.get('year')) == str(year):
            a, p = int(e['src']), int(e['dst'])
            if p < N_ROWS:
                ap_edges.append((a, p))
                authors_set.add(a)
                papers_set.add(p)

    # --- AUTHOR-AUTHOR ---
    for e in read_jsonl(AA_PATH):
        if str(e.get('year')) == str(year):
            a1, a2 = int(e['a1']), int(e['a2'])
            aa_edges.append((a1, a2))
            authors_set.add(a1)
            authors_set.add(a2)

    # --- PAPER-PAPER ---
    for e in read_jsonl(PP_PATH):
        if str(e.get('year')) == str(year):
            p1, p2 = int(e['src']), int(e['dst'])
            if p1 < N_ROWS and p2 < N_ROWS:
                if p1 in papers_set and p2 in papers_set:
                    pp_edges.append((p1, p2))

    if not authors_set and not papers_set:
        return None

    # Mapping global → local
    authors_list = sorted(authors_set)
    papers_list  = sorted(papers_set)

    a_g2l = {a: i for i, a in enumerate(authors_list)}
    p_g2l = {p: i for i, p in enumerate(papers_list)}

    # =========================
    # DATA OBJECT
    # =========================
    data = HeteroData()

    data['paper'].x = torch.from_numpy(
        paper_embeddings[papers_list].copy()
    )

    data['author'].x = torch.zeros(
        (len(authors_list), AUTHOR_DIM),
        dtype=torch.float32
    )

    data['author'].node_id = torch.tensor(authors_list)
    data['paper'].node_id  = torch.tensor(papers_list)

    # --- A-P ---
    if ap_edges:
        src = [a_g2l[a] for a, _ in ap_edges]
        dst = [p_g2l[p] for _, p in ap_edges]

        data['author', 'writes', 'paper'].edge_index = torch.tensor([src, dst])
        data['paper', 'rev_writes', 'author'].edge_index = torch.tensor([dst, src])

    # --- A-A ---
    if aa_edges:
        s = [a_g2l[a1] for a1, a2 in aa_edges]
        d = [a_g2l[a2] for a1, a2 in aa_edges]

        data['author', 'coauthor', 'author'].edge_index = torch.tensor([s + d, d + s])

    # --- P-P ---
    if pp_edges:
        s = [p_g2l[p1] for p1, _ in pp_edges]
        d = [p_g2l[p2] for _, p2 in pp_edges]

        data['paper', 'cites', 'paper'].edge_index = torch.tensor([s, d])

    return data

# =========================
# MAIN LOOP
# =========================
print("\n🚀 Génération des snapshots...")

for year in tqdm(range(YEAR_START, YEAR_END + 1)):
    out_path = f'{SNAPSHOT_DIR}/snapshot_{year}.pt'

    data = build_snapshot(year)

    if data is None:
        print(f'⚠️ {year}: snapshot vide')
        torch.save(HeteroData(), out_path)
    else:
        torch.save(data, out_path)
        print(f'✅ {year}: {data["author"].num_nodes:,} auteurs | {data["paper"].num_nodes:,} papers')

print("\n🎉 Snapshots terminés !")